# construção de Machine Learning para a ONG Passos Mágicos.
Os modelos de Machine Learning que serão aqui construídos serão utilizados para prever:
- se o aluno esta elegível para a obtenção de uma BOLSA DE ESTUDOS ou
- se o aluno se esta elegível para alcance do PONTO DE VIRADA.



In [ ]:
import re
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
from scipy.stats import gaussian_kde
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('df_modelo.csv', sep=';',encoding="latin-1")
df.head()

In [ ]:
df['PONTO_VIRADA_2021'].value_counts()

In [ ]:
df_analysis = df[['NOME', 'PONTO_VIRADA_2021']].isin(['#NULO!'])
df_analysis[df_analysis['PONTO_VIRADA_2021'] == True]
# will remove these in a function later on, with NaN

In [ ]:
indicadores = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']

In [ ]:
def extract_columns_for_year(dataframe, year):
    regex_pattern = re.compile(f'.*_{year}$')
    columns_for_year = [column for column in dataframe.columns if regex_pattern.match(column) or column == 'NOME']
    filtered_dataframe = dataframe[columns_for_year].copy()
    filtered_dataframe['ANO'] = year
    filtered_dataframe.columns = [re.sub(f'_{year}$', '', column) for column in filtered_dataframe.columns]

    return filtered_dataframe

def clean_dataframe(dataframe):
    # Drop rows where all columns except 'NOME' and 'ANO' are NaN
    temp_df = dataframe.dropna(subset=dataframe.columns.difference(['NOME', 'ANO']), how='all')
    # Remove rows that are entirely NaN
    temp_df = temp_df[~temp_df.isna().all(axis=1)]
    # remove rows where we have #NULO! in any of the columns
    temp_df = temp_df[~temp_df.isin(['#NULO!']).any(axis=1)]
    return temp_df.dropna(subset=['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN'], how='all')

def process_yearly_data(dataframe, year: int) -> pd.DataFrame:
    filtered_dataframe = extract_columns_for_year(dataframe, year)
    year_specific_df = filtered_dataframe.query('ANO == @year')

    # Convert indicator columns to numeric, forcing errors to NaN
    indicadores = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN']
    year_specific_df[indicadores] = year_specific_df[indicadores].apply(lambda column: pd.to_numeric(column, errors='coerce'))

    # Generate a cleaned version of the dataframe without NaNs
    cleaned_year_df = clean_dataframe(year_specific_df)

    return cleaned_year_df

In [ ]:
# quebra o df original por ano
df_2020_cleaned = process_yearly_data(df, 2020)
df_2021_cleaned = process_yearly_data(df, 2021)
df_2022_cleaned = process_yearly_data(df, 2022)

In [ ]:
busca_string_invalida = df_2020_cleaned.map(lambda x: x == '#NULO!')
df_2020_cleaned[busca_string_invalida.any(axis=1)]

In [ ]:
busca_string_invalida = df_2021_cleaned.map(lambda x: x == '#NULO!')
df_2021_cleaned[busca_string_invalida.any(axis=1)]

In [ ]:
busca_string_invalida = df_2022_cleaned.map(lambda x: x == '#NULO!')
df_2022_cleaned[busca_string_invalida.any(axis=1)]

# Conferindo

In [ ]:
df_concat = pd.concat([df_2020_cleaned, df_2021_cleaned, df_2022_cleaned], ignore_index=True)


df_2020_cleaned.to_csv('data_2020_old.csv', sep=';', index=False)
df_2021_cleaned.to_csv('data_2021_old.csv', sep=';', index=False)
df_2022_cleaned.to_csv('data_2022_old.csv', sep=';', index=False)
df_concat.to_csv('data_full.csv', sep=';', index=False)

In [ ]:
df_2020_cleaned

In [ ]:
df_2020_cleaned['PEDRA'].value_counts()


In [ ]:
df_2020_cleaned['ANO'].value_counts()

In [ ]:
df_2020_cleaned['INSTITUICAO_ENSINO_ALUNO'].value_counts()


In [ ]:
df_2022_cleaned


In [ ]:
df_2022_cleaned.INDICADO_BOLSA


In [ ]:
df_2022_cleaned['INDICADO_BOLSA'].value_counts()


In [ ]:
df_2022_cleaned

# Descrição dos Indicadores e do índice INDE que serão utilizados em ambos os modelos.
  IAA: Indicador de Auto Avalição – calculado com base na média das notaa
obtidas em questionário de autoavaliação individual.

  IEG: Indicador de Engajamento – calculado com base nos registros de entrega de lição de casa e de atividades de voluntariado.

  IPS: Indicador Psicosocial – calculado com base na média das notaa obtidas em questionário individual de avaliação elaborado pelas psicólogas.

  IDA: Indicador de Desempenho Acadêmico - calculado com base nas notas das provas aplicadas pela Passos Mágicos e pela média geral das universitárias.

  IPP: Indicador Psico Pedagógico - calculado com base em questionário individual de avaliação elaborado pelas pedagogos e professores.

  IPV: Indicador de Ponto de Virada - calculado com base em questionário individual de avaliação elaborado pelas pedagogos e professores.

  IAN: Indicador de Adequação de Nível - calculado com base na fase efetiva em que o aluno se encontra e na fase ideal em ele deveria estar.

  INDE: Índice de Desenvolvimento Educacional
  
  PONTO VIRADA: Indica se o aluno atingiu o ponto de virada, ou seja, avançou e seu nível de performance e desenvolvimento educacional, representado pela Pedras.




In [ ]:
df_bolsa_predict = df_2022_cleaned[['IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN', 'INDE', 'INDICADO_BOLSA']]
df_bolsa_predict

In [ ]:
label_encoder = LabelEncoder()
df_bolsa_predict['INDICADO_BOLSA'] = label_encoder.fit_transform(df_bolsa_predict['INDICADO_BOLSA'])
df_bolsa_predict

# Contrução do Modelo Preditivo para prever se o aluno vai alcançar o PONTO DE VIRADA

**Data Preparation:**


Split the data into features (X) and target (y). Normalize the features to ensure that all the input data has a similar scale. Split the data into training and testing sets.


**Handling Imbalance:**

Use techniques like oversampling (e.g., SMOTE) or adjusting class weights during model training.


**Model Creation:**

Build a neural network using a library like TensorFlow/Keras. Define the input layer, hidden layers, and output layer. Use an appropriate activation function for the output layer (e.g., sigmoid for binary classification).


**Model Training:**

Compile the model using an optimizer (e.g., Adam) and a suitable loss function (e.g., binary crossentropy). Train the model on the training data.


**Model Evaluation:**

Evaluate the model's performance on the test set using metrics like accuracy, precision, recall, and the ROC-AUC score.

In [ ]:
df_all = pd.concat([df_2020_cleaned, df_2021_cleaned, df_2022_cleaned])
df_ponto_virada_to_predict = df_all[['IAA', 'IEG', 'IPS', 'IDA', 'IPP', 'IPV', 'IAN', 'INDE', 'PONTO_VIRADA']]
df_ponto_virada_to_predict.dropna(inplace=True)
df_ponto_virada_to_predict

In [ ]:
label_encoder = LabelEncoder()
df_ponto_virada_to_predict['PONTO_VIRADA'] = label_encoder.fit_transform(df_ponto_virada_to_predict['PONTO_VIRADA'])
df_ponto_virada_to_predict

In [ ]:
df_ponto_virada_to_predict.PONTO_VIRADA.value_counts()

# Handling Correlated Features

In [ ]:
# Assuming df is your DataFrame
from matplotlib import pyplot as plt
import seaborn as sns


correlation_matrix = df_ponto_virada_to_predict.corr()

# Set a threshold for correlation
threshold = 0.8

# Identify highly correlated pairs
highly_correlated = []
for column in correlation_matrix.columns:
    for index in correlation_matrix.index:
        if abs(correlation_matrix.loc[index, column]) > threshold and column != index:
            highly_correlated.append((column, index))

# Determine columns to drop, ensuring each column is checked before dropping
columns_to_drop = set()

for column1, column2 in highly_correlated:
    # Add column2 to drop if it's still in the DataFrame
    if column2 in df_ponto_virada_to_predict.columns:
        columns_to_drop.add(column2)

# Drop the correlated columns from the DataFrame
df_ponto_virada_to_predict_reduced = df_ponto_virada_to_predict.drop(columns=columns_to_drop, errors='ignore')

# Visualize the reduced correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df_ponto_virada_to_predict_reduced.corr(), annot=True, cmap='coolwarm')
plt.title('Reduced Correlation Matrix Heatmap')
plt.show()

# Print the columns that were removed and the remaining columns
print(f"Columns removed: {columns_to_drop}")
print(f"Remaining columns: {df_ponto_virada_to_predict_reduced.columns}")

# Salvando os datasets com os dados que serão usados para treinar os modelos da BOLSA e do PONTO DE VIRADA


In [ ]:
df_bolsa_predict.to_csv('output/df_bolsa_predict.csv', sep=',', index=False)
df_ponto_virada_to_predict.to_csv('output/df_ponto_virada_to_predict.csv', sep=',', index=False)

In [ ]:
df_ponto_virada_to_predict_reduced.PONTO_VIRADA.value_counts()

In [ ]:
df_ponto_virada_to_predict_reduced

# Normalizando os dados (Scaler) e gerando os arrays X e Y


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

X = df_ponto_virada_to_predict_reduced.drop('PONTO_VIRADA', axis=1)
Y = df_ponto_virada_to_predict_reduced['PONTO_VIRADA']

# Convert categorical features to numeric using one-hot encoding, if needed
X = pd.get_dummies(X)

# Normalize the features
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Ensure X and Y are numpy arrays
X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.float32)

# Verify shape and type of the arrays after conversion
print(f"X shape: {X.shape}, X dtype: {X.dtype}")
print(f"Y shape: {Y.shape}, Y dtype: {Y.dtype}")
print("Sample of X data:\n", X[:5])
print("Sample of Y data:\n", Y[:5])

# Check for any remaining non-numeric or NaN values
assert np.issubdtype(X.dtype, np.number), "X contains non-numeric values."
assert not np.isnan(X).any(), "X contains NaN values."
assert np.issubdtype(Y.dtype, np.number), "Y contains non-numeric values."
assert not np.isnan(Y).any(), "Y contains NaN values."


# Check for any non-numeric or NaN values
print(f"X dtype: {X.dtype}, Y dtype: {Y.dtype}")
print(f"Any NaN in X: {np.isnan(X).any()}, Any NaN in Y: {np.isnan(Y).any()}")

X

# Gerando os arrays de treino e de teste para treinar e testar o modelo.


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
print(len(Y_train))

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, Y_train)

In [ ]:
df_ponto_virada_to_predict_reduced['PONTO_VIRADA'].value_counts()


In [ ]:
X_test.dtype


# - "Inicializando", "treinando", "fazendo previsões com os dados de teste" e "avaliando" o modelo de classificação RANDOM FOREST a ser usado nas previsões de PONTO DE VIRADA.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Random Forest model
model = RandomForestClassifier(random_state=42)

# Train the model
model.fit(X_train_res, y_train_res)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(Y_test, y_pred)
report = classification_report(Y_test, y_pred)

In [ ]:
accuracy


In [ ]:
print(report)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

# Generate the confusion matrix
cm = confusion_matrix(Y_test, y_pred)

# Print the confusion matrix
print("Confusion Matrix:\n", cm)

# Optionally, use ConfusionMatrixDisplay for a visual representation
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots()
disp.plot(ax=ax, cmap=plt.cm.Blues)

# Analisando a importância de cada um dos Indicadores no Modelo Randon Forest


In [ ]:
import matplotlib.pyplot as plt

feature_names = df_ponto_virada_to_predict_reduced.drop('PONTO_VIRADA', axis=1).columns

# Get feature importances
importances = model.feature_importances_
features = feature_names

# Create a DataFrame for visualization
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
feature_importance_df.to_csv('output/ponto_virada_feature_importance.csv', sep=',', index=False)

# Plot the feature importances
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importância')
plt.ylabel('Indicador')
plt.title('Importância dos Indicadores no Modelo Randon Forest')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(model.estimators_[0], feature_names=feature_names, class_names=pd.Index(['0', '1']), filled=True)
plt.title('Tree Structure')
plt.show()

In [ ]:
# get row where student passed
df_ponto_virada_to_predict_reduced.query('PONTO_VIRADA == 1')

# Simulando casos para testar o modelo preditivo de alcance do PONTO DE VIRADA.


In [ ]:
import numpy as np

# Create mock data with the same feature names as the original data
mock_data = pd.DataFrame({
    'IAA': [8.33],
    'IPS': [4.375],
    'IPP': [8.75],
    'IPV': [8.95],
    'IAN': [10.0],
})

# Preprocess the mock data using the scaler
mock_data_scaled = scaler.transform(mock_data)

# Predict using the loaded model
mock_prediction = model.predict(mock_data_scaled)

if mock_prediction[0] == 1:
    print("O aluno apresenta alta probabilidade de subir de nível.")
else:
    print("O aluno apresenta baixa probabilidade de subir de nível.")

# Salvando o modelo do Ponto de Virada no formato .pkl para fazer o DEPLOY no Streamlit

In [ ]:
import joblib

# Save the model
joblib.dump(scaler, 'random_forest_ponto_virada_scaler.pkl')
joblib.dump(model, 'random_forest_ponto_virada_model.pkl')

# Construção do modelo preditivo p/saber se o aluno pode receber uma BOLSA DE ESTUDOS.


Data Preparation:

Split the data into features (X) and target (y). Normalize the features to ensure that all the input data has a similar scale. Split the data into training and testing sets.


Handling Imbalance:

Use techniques like oversampling (e.g., SMOTE) or adjusting class weights during model training.


Model Creation:

Build a neural network using a library like TensorFlow/Keras. Define the input layer, hidden layers, and output layer. Use an appropriate activation function for the output layer (e.g., sigmoid for binary classification).


Model Training:

Compile the model using an optimizer (e.g., Adam) and a suitable loss function (e.g., binary crossentropy). Train the model on the training data.


Model Evaluation:

Evaluate the model's performance on the test set using metrics like accuracy, precision, recall, and the ROC-AUC score.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
df_perfil_receber_bolsa_estudos = pd.read_csv('output/df_bolsa_predict.csv')
df_perfil_receber_bolsa_estudos.head()

In [ ]:
columns = df_perfil_receber_bolsa_estudos.columns.difference(['INDICADO_BOLSA'])
good_grades = np.random.uniform(8, 10, size=(200, len(columns)))
good_grades = pd.DataFrame(good_grades, columns=columns)
good_grades['INDICADO_BOLSA'] = 1
good_grades.head()

In [ ]:
bad_grades = np.random.uniform(0, 5, size=(400, len(columns)))
bad_grades = pd.DataFrame(bad_grades, columns=columns)
bad_grades['INDICADO_BOLSA'] = 0
bad_grades

In [ ]:
df_perfil_receber_bolsa_estudos_final = pd.concat([df_perfil_receber_bolsa_estudos, good_grades, bad_grades], ignore_index=True)

df_perfil_receber_bolsa_estudos_final

In [ ]:
print(df_perfil_receber_bolsa_estudos_final['INDICADO_BOLSA'].value_counts())


In [ ]:
# features
X = df_perfil_receber_bolsa_estudos_final[df_perfil_receber_bolsa_estudos_final.columns.difference(['INDICADO_BOLSA'])]
# targets
y = df_perfil_receber_bolsa_estudos_final['INDICADO_BOLSA']

X.size, X_train.size

In [ ]:
# Split the resampled data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y)

In [ ]:
y_test.value_counts()

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
scaler.fit(X_test)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train.shape, X_test.shape


# "Inicializando", "treinando", "fazendo previsões com os dados de teste" e "avaliando" o modelo de classificação RANDOM FOREST a ser usado nas previsões de concessão de BOLSA DE ESTUDOS.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Reshape y_train and y_test for binary classification
y_train = np.asarray(y_train).reshape((-1, 1))
y_test = np.asarray(y_test).reshape((-1, 1))

# Define input and output shapes
input_shape = X_train.shape[1]    # Variáveis de entrada
output_shape = y_train.shape[1]   # Classe preditora
batch_size = 32

# Step 3: Model Creation with Input layer
model = Sequential([
    Dense(batch_size, input_shape=(input_shape,),activation='relu'),
    Dense(8, activation='relu'),
    Dropout(0.5),
    Dense(output_shape, activation='sigmoid')
])

# Step 4: Model Training
epoch = 2000
learning_rate = 0.00002
model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=20, verbose=1, mode='auto')

history = model.fit(X_train, y_train, epochs=epoch, shuffle=True, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping],  verbose=0)

# Step 5: Model Evaluation
y_pred = model.predict(X_test)
y_pred_class = (y_pred > 0.5).astype(int)

# Calculate metrics
classification_rep = classification_report(y_test, y_pred_class)
conf_matrix = confusion_matrix(y_test, y_pred_class)
roc_auc = roc_auc_score(y_test, y_pred)

classification_rep, conf_matrix, roc_auc

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Predicted 0', 'Predicted 1'], yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
model_training_data_hist = pd.DataFrame(history.history)
model_training_data_hist

In [ ]:
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

# Use the length of the training history to set the range
epochs_range = range(len(train_acc))

# Plot Acurácia
plt.figure(figsize=(20, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_acc, label='Acurácia de Treinamento')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação')
plt.legend(loc='lower right')
plt.title('Acurácia de treino e teste')

# Plot Erro de treinamento
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_loss, label='Erro de treinamento')
plt.plot(epochs_range, val_loss, label='Erro de Validação')
plt.legend(loc='upper right')
plt.title('Erro de treinamento vs validação')
plt.show()

In [ ]:
df_perfil_receber_bolsa_estudos

In [ ]:
scaler.feature_names_in_


In [ ]:
# Create mock data
mock_data = pd.DataFrame({
    'IAA': [10],
    'IEG': [9.16],
    'IPS': [7.50],
    'IDA': [8.89],
    'IPP': [8.12],
    'IPV': [8.21],
    'IAN': [8.0],
    'INDE': [8.95]
})

# Ensure the mock data columns match the original data used to fit the scaler
mock_data = mock_data[scaler.feature_names_in_]

# Preprocess the mock data using the scaler
mock_data_scaled = scaler.transform(mock_data)

# Predict using the loaded model
mock_prediction = model.predict(mock_data_scaled)

mock_prediction.flatten().round()

In [ ]:
# Create mock data
mock_data = pd.DataFrame({
    'IAA': [3],
    'IEG': [4.16],
    'IPS': [7.50],
    'IDA': [0.89],
    'IPP': [8.12],
    'IPV': [8.21],
    'IAN': [2.0],
    'INDE': [8.95]
})

# Ensure the mock data columns match the original data used to fit the scaler
mock_data = mock_data[scaler.feature_names_in_]

# Preprocess the mock data using the scaler
mock_data_scaled = scaler.transform(mock_data)

# Predict using the loaded model
mock_prediction = model.predict(mock_data_scaled)

mock_prediction.flatten().round()

# Salvando o modelo BOLSA DE ESTUDOS no formato .pkl para fazer o DEPLOY no Streamlit


In [ ]:
import joblib

model_training_data_hist.to_csv('model_training_data_hist.csv', index=None, sep=';')

# Save the model
joblib.dump(scaler, 'perfil_receber_bolsa_estudos_scaler.pkl')
joblib.dump(model, 'perfil_receber_bolsa_estudos_model.pkl')

O modelo preditivo desenvolvido para ONG Passos Mágicospara prever a probabilidade que um aluno terá de avançar
        para a próxima fase com base nos indicadores de desempenho por ele alcançados até então,  foi elaborado com base em um algoritmo
        de Random Forest, conhecido por sua robustez em tarefas de classificação.

Razões pelas quais foi escolhido o modelo Random Forest:

        - **Interpretabilidade**: os modelos Random Forest são fáceis de interpretar e entender.

        - **Importância do recurso**: os modelos Random Forest fornecem importância dos atributos ou indicadores, o que ajuda a compreender o impacto de cada indicador na previsão.

        - **Robustez**: os modelos Random Forest são robustos ao overfitting e ao ruído nos dados.

        - **Relacionamentos não lineares**: os modelos Random Forest podem capturar relacionamentos não lineares entre recursos e a variável de destino.

        - **Lida com dados ausentes e conjuntos de dados desequilibrados**: os modelos Random Forest podem lidar com dados ausentes e conjuntos de dados desequilibrados de maneira eficaz. Pode ser ajustado com pesos de classe ou combinado com técnicas como SMOTE para lidar com classes desequilibradas

# indicadores de desempenho utilizados no modelo são os seguintes:

IPV - Indicador do Ponto de Virada, calculado com base em questionário individual de avaliação elaborado pelas pedagogos e professores.

IPP - Indicador Psicopedagógico, calculado com base em questionário individual de avaliação elaborado pelas pedagogos e professores.

IAA - Indicador de Autoavaliação, calculado com base em questionário de autoavaliação individual.

IPS - Indicador Psicossocial, calculado com base em questionário individual de avaliação elaborado pelas psicólogas.

IAN - Indicador de Adequação de Nível, calculado com base na fase efetiva em que o aluno se encontra e na fase ideal em ele deveria estar.


# Codigo pra subir o modelo no site

In [ ]:
import joblib
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go


# Setting the title of the Streamlit app
st.title('**:green[Modelo Preditivo - Ponto de Virada]**')

# Providing a brief description of the app and the model it uses
st.markdown(
        """
        O modelo preditivo desenvolvido para a **:orange[ONG Passos Mágicos]** para prever a probabilidade que um aluno terá de avançar
        para a próxima fase com base nos indicadores de desempenho por ele alcançados até então,  foi elaborado com base em um algoritmo
        de Random Forest, conhecido por sua robustez em tarefas de classificação.

        """, unsafe_allow_html=True)

# Loading the pre-trained model and scaler from disk using joblib
with st.container():
    model_path = "random_forest_ponto_virada_model.pkl"
    scaler_path = "random_forest_ponto_virada_scaler.pkl"
    feature_importance_df = pd.read_csv(
        "output/ponto_virada_feature_importance.csv", sep=","
    )

    # Load the model and scaler using joblib
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)


    st.success("Modelo preditivo carregado com sucesso!")

    st.markdown(
        """
        <br>Razões pelas quais foi escolhido o modelo Random Forest:
        - **Interpretabilidade**: os modelos Random Forest são fáceis de interpretar e entender.
        - **Importância do recurso**: os modelos Random Forest fornecem importância dos atributos ou indicadores, o que ajuda a compreender o impacto de cada indicador na previsão.
        - **Robustez**: os modelos Random Forest são robustos ao overfitting e ao ruído nos dados.
        - **Relacionamentos não lineares**: os modelos Random Forest podem capturar relacionamentos não lineares entre recursos e a variável de destino.
        - **Lida com dados ausentes e conjuntos de dados desequilibrados**: os modelos Random Forest podem lidar com dados ausentes e conjuntos de dados desequilibrados de maneira eficaz. Pode ser ajustado com pesos de classe ou combinado com técnicas como SMOTE para lidar com classes desequilibradas

        """, unsafe_allow_html=True)

    st.markdown(
        """
        <br>Os indicadores de desempenho utilizados no modelo são os seguintes:

    ###### <font color='lightblue'>IPV - Indicador do Ponto de Virada</font>, calculado com base em questionário individual de avaliação elaborado pelas pedagogos e professores.
    ###### <font color='lightblue'>IPP - Indicador Psicopedagógico</font>, calculado com base em questionário individual de avaliação elaborado pelas pedagogos e professores.
    ###### <font color='lightblue'>IAA - Indicador de Autoavaliação</font>, calculado com base em questionário de autoavaliação individual.
    ###### <font color='lightblue'>IPS - Indicador Psicossocial</font>, calculado com base em questionário individual de avaliação elaborado pelas psicólogas.
    ###### <font color='lightblue'>IAN - Indicador de Adequação de Nível</font>, calculado com base na fase efetiva em que o aluno se encontra e na fase ideal em ele deveria estar.

        """, unsafe_allow_html=True)

    st.markdown(
        """
        O gráfico de importância de cada indicador abaixo mostra a importância, peso ou influência de cada um deles no modelo. Quanto maior o valor, mais importante é o indicador na previsão da probabilidade de avanço de fase do aluno.

        """, unsafe_allow_html=True)

    with st.container():

        # Plot using Plotly
        fig = go.Figure()

        fig.add_trace(
            go.Bar(
                x=feature_importance_df['Importance'],
                y=feature_importance_df['Feature'],
                orientation='h',
                marker=dict(color='lightskyblue'),  # Set the color to light green
                name="Feature Importance",
            )
        )

        fig.update_layout(
            title="Importância dos Indicadores no modelo Random Forest",
            xaxis_title="Importância",
            yaxis_title="Indicador",
            yaxis=dict(
                autorange="reversed"  # To invert y-axis like plt.gca().invert_yaxis()
            ),
            height=600,
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1,
            ),
        )

        # Display the plot in Streamlit
        st.plotly_chart(fig)

        # Explanation of how to use the controls and what the model predicts
    st.markdown(
        """
Para fazer uso do modelo, é necessário informar nos campos abaixo os indicadores de desempenho alcançados pelo aluno. A seguir, para executar o modelo e descobrir se o aluno passará para a próxima fase com base em seu desempenho, é necessário clicar no botão **:orange[🎯Modelo, qual é a sua previsão?]**

**:red[ATENÇÂO]**: Os campos precisam ser preenchidos lentamente, caso contrário o modelo não reconhecerá a matriz de entrada no formato requerido.""", unsafe_allow_html=True)

    # Collecting input values for various student performance indicators from the user
    with st.container():
        col0, col1, col2, col3, col4 = st.columns(5)

        # Collecting input for IPV indicator
        with col0:
            indicator_ipv = st.number_input(
                label="**:blue[IPV]**",
                key="ipv",
                min_value=0.0,
                max_value=10.0,
                value=0.0,
                step=0.1,
                format="%.2f",
            )


        # Collecting input for IPP indicator
        with col1:
            indicator_ipp = st.number_input(
                label="**:blue[IPP]**",
                key="ipp",
                min_value=0.0,
                max_value=10.0,
                value=0.0,
                step=0.1,
                format="%.2f",
            )


        # Collecting input for IAA indicator
        with col2:
            indicator_iaa = st.number_input(
                label="**:blue[IAA]**",
                key="iaa",
                min_value=0.0,
                max_value=10.0,
                value=0.0,
                step=0.1,
                format="%.2f",
            )


        # Collecting input for IPS indicator
        with col3:
            indicator_ips = st.number_input(
                label="**:blue[IPS]**",
                key="ips",
                min_value=0.0,
                max_value=10.0,
                value=0.0,
                step=0.1,
                format="%.2f",
            )


        # Collecting input for IAN indicator
        with col4:
            indicator_ian = st.number_input(
                label="**:blue[IAN]**",
                key="ian",
                min_value=0.0,
                max_value=10.0,
                value=0.0,
                step=0.1,
                format="%.2f",
            )



    # Creating a DataFrame from the input values to feed into the model
    student_data = pd.DataFrame(
        {
            'IAA': indicator_iaa,
            'IPS': indicator_ips,
            'IPP': indicator_ipp,
            'IPV': indicator_ipv,
            'IAN': indicator_ian,
        },
        index=[0],
    )

    student_data_ordered = student_data.reindex(["IPV","IPP", "IAA","IPS","IAN"], axis=1)

    # Adding a button to trigger the model prediction
    if st.button("**:orange[🎯Modelo, qual é a sua previsão?]**", key="btn_predict_mlp"):
        with st.spinner("Processing..."):
            st.subheader(":blue[Dados de Entrada do Modelo]", divider="blue")
            st.dataframe(student_data_ordered, hide_index=True)

            st.subheader(":blue[Resposta do Modelo]", divider="blue")

            # Scaling the input data using the loaded scaler
            student_data_scaled = scaler.transform(student_data)

            # Predicting whether the student will pass to the next grade using the pre-trained model
            prediction = model.predict(student_data_scaled)

            # Displaying the result of the prediction
            if prediction.round()[0] > 0:
                st.success(
                    ":white_check_mark: O aluno apresenta alta probabilidade de alcançar o ponto de virada e passar para um nível de desenvolvimento mais elevado. 🏆"
                )
            else:
                st.error(
                    ":x: O aluno apresenta baixa probabilidade de alcançar o ponto de virada e passar para um nível de desenvolvimento mais elevado. 😞"
                )